# 01 - SfM + MVS + Render

This notebook builds a sparse SfM reconstruction (COLMAP) for one dataset cluster, optionally
runs dense MVS, and renders a quick preview.

Requirements:
- COLMAP installed and available on PATH (or edit COLMAP_BIN).

Notes:
- Uses `cache/clusters.csv` to select a scene cluster.
- Uses `cache/<dataset>/pairs_topk_K30.npz` to build a pair list for inspection.
- Matching defaults to exhaustive for small clusters.


In [7]:
from pathlib import Path
import csv
import json
import os
import shutil
import subprocess
import numpy as np

from pathlib import Path
def detect_root():
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "pyproject.toml").exists() or (cand / "cache").exists():
            return cand
    return here
ROOT = Path("/home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/") # Path("/home/ubuntu/ml/")
DATASET = "ETs"
CLUSTER = "cluster_0001"
# CACHE_ROOT = ROOT / "temp-del/cache"
CACHE_ROOT = ROOT / "cache"
OUT_ROOT = ROOT / "outputs" / f"{DATASET}_{CLUSTER}"
COLMAP_BIN = shutil.which("colmap") or "colmap"

print("ROOT:", ROOT)
print("COLMAP_BIN:", COLMAP_BIN)

ROOT: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025
COLMAP_BIN: colmap


In [9]:
def load_cluster_image_ids(dataset: str, cluster: str):
    cluster_path = CACHE_ROOT / "clusters.csv"
    ids = []
    with cluster_path.open("r", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row["dataset"] == dataset and row["scene"] == cluster:
                ids.append(row["image_id"])
    return ids

cluster_ids = load_cluster_image_ids(DATASET, CLUSTER)
print("cluster size:", len(cluster_ids))
if not cluster_ids:
    raise RuntimeError("Cluster is empty. Check DATASET/CLUSTER and cache/clusters.csv.")


cluster size: 7


In [10]:
meta_path = CACHE_ROOT / DATASET / "meta.json"
meta = json.loads(meta_path.read_text())
image_ids = meta["image_ids"]
paths = meta["paths"]
id_to_path = dict(zip(image_ids, paths))

raw_paths = [Path(id_to_path[i]) for i in cluster_ids]
image_paths = [p if p.is_absolute() else (ROOT / p) for p in raw_paths]
image_dirs = {p.parent for p in image_paths}
if len(image_dirs) != 1:
    raise RuntimeError(f"Expected one image directory, got: {image_dirs}")
image_dir = next(iter(image_dirs))

print("image_dir:", image_dir)
print("num images:", len(image_paths))

image_dir: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/data/train/ETs
num images: 7


In [11]:
OUT_ROOT.mkdir(parents=True, exist_ok=True)
db_path = OUT_ROOT / "colmap.db"
sparse_dir = OUT_ROOT / "sparse"
dense_dir = OUT_ROOT / "dense"
image_list_path = OUT_ROOT / "image_list.txt"
pairs_path = OUT_ROOT / "pairs.txt"

# Copy cluster images into a dedicated folder to avoid relying on image_list_path,
# which is not supported in some COLMAP builds.
work_image_dir = OUT_ROOT / "images"
COPY_IMAGES = True
if COPY_IMAGES:
    work_image_dir.mkdir(parents=True, exist_ok=True)
    for p in image_paths:
        dst = work_image_dir / p.name
        if not dst.exists():
            shutil.copy2(p, dst)
else:
    work_image_dir = image_dir

with image_list_path.open("w") as f:
    for p in image_paths:
        f.write(p.name + "\n")

print("work_image_dir:", work_image_dir)
print("image_list_path:", image_list_path)


work_image_dir: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ETs_cluster_0001/images
image_list_path: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ETs_cluster_0001/image_list.txt


In [12]:
# Build a pair list from pairs_topk_K30.npz (optional)
pairs_npz = CACHE_ROOT / DATASET / "pairs_topk_K30.npz"
pairs = []
if pairs_npz.exists():
    data = np.load(pairs_npz)
    a = data["a"]
    b = data["b"]
    cluster_set = set(cluster_ids)
    for i, j in zip(a, b):
        ida = image_ids[int(i)]
        idb = image_ids[int(j)]
        if ida in cluster_set and idb in cluster_set:
            pa = Path(id_to_path[ida]).name
            pb = Path(id_to_path[idb]).name
            pairs.append((pa, pb))

with pairs_path.open("w") as f:
    for a, b in pairs:
        f.write(f"{a} {b}\n")

print("pairs_path:", pairs_path)
print("num pairs:", len(pairs))


pairs_path: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ETs_cluster_0001/pairs.txt
num pairs: 21


Note: COLMAP's exhaustive matcher ignores `pairs.txt`.
If you want to restrict matching to pairs, use a custom matcher (e.g., hloc) or a
pairwise-matching script, then import matches into COLMAP.


In [13]:
def run(cmd):
    print(" ".join(cmd))
    env = os.environ.copy()
    env["QT_QPA_PLATFORM"] = "offscreen"
    env["LIBGL_ALWAYS_SOFTWARE"] = "1"
    if shutil.which("xvfb-run"):
        subprocess.run(["xvfb-run", "-a"] + cmd, check=True, env=env)
    else:
        subprocess.run(cmd, check=True, env=env)

RUN_COLMAP = True
if not shutil.which("colmap"):
    print("COLMAP not found. Install it and re-run this cell.")
    RUN_COLMAP = False


COLMAP not found. Install it and re-run this cell.


In [14]:
# 1) Feature extraction
if RUN_COLMAP:
    if db_path.exists():
        db_path.unlink()
    run([
        "colmap", "feature_extractor",
        "--database_path", str(db_path),
        "--image_path", str(work_image_dir),
        "--ImageReader.single_camera", "1",
        "--ImageReader.camera_model", "SIMPLE_RADIAL",
        "--SiftExtraction.use_gpu", "0",
    ])


In [15]:
# 2) Feature matching
# For small clusters, exhaustive is fine. For larger clusters, consider spatial/sequential.
if RUN_COLMAP:
    run([
        "colmap", "exhaustive_matcher",
        "--database_path", str(db_path),
        "--SiftMatching.guided_matching", "1",
        "--SiftMatching.use_gpu", "0",
    ])


In [16]:
# 3) Sparse reconstruction
if RUN_COLMAP:
    sparse_dir.mkdir(parents=True, exist_ok=True)
    run([
        "colmap", "mapper",
        "--database_path", str(db_path),
        "--image_path", str(work_image_dir),
        "--output_path", str(sparse_dir),
        "--Mapper.ba_refine_principal_point", "0",
    ])


In [17]:
# 4) Convert sparse model to PLY for quick visualization
sparse_model = sparse_dir / "0"
sparse_ply = OUT_ROOT / "sparse_points.ply"
if RUN_COLMAP and sparse_model.exists():
    run([
        "colmap", "model_converter",
        "--input_path", str(sparse_model),
        "--output_path", str(sparse_ply),
        "--output_type", "PLY",
    ])
    print("sparse_ply:", sparse_ply)


In [ ]:
# 5) dense reconstruction
RUN_DENSE = False
if RUN_COLMAP and RUN_DENSE:
    dense_dir.mkdir(parents=True, exist_ok=True)
    run([
        "colmap", "image_undistorter",
        "--image_path", str(work_image_dir),
        "--input_path", str(sparse_model),
        "--output_path", str(dense_dir),
        "--output_type", "COLMAP",
    ])
    run([
        "colmap", "patch_match_stereo",
        "--workspace_path", str(dense_dir),
        "--workspace_format", "COLMAP",
        "--PatchMatchStereo.geom_consistency", "true",
    ])
    run([
        "colmap", "stereo_fusion",
        "--workspace_path", str(dense_dir),
        "--workspace_format", "COLMAP",
        "--input_type", "geometric",
        "--output_path", str(OUT_ROOT / "dense_points.ply"),
    ])


In [ ]:
import open3d as o3d

pcd = o3d.io.read_point_cloud(str(sparse_ply))

out_path = sparse_ply.with_name("sparse_copy.ply")
o3d.io.write_point_cloud(str(out_path), pcd)
print("Wrote:", out_path)


[Open3D WARNING] Read PLY failed: unable to open file: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ETs_cluster_0001/sparse_points.ply
[Open3D WARNING] Write PLY failed: point cloud has 0 points.
Wrote: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ETs_cluster_0001/sparse_copy.ply


RPly: Unable to open file


In [20]:
from pathlib import Path

cwd = Path.cwd()
print("CWD:", cwd)

# search up one level (repo root-ish) and below
root = cwd.parent
ply_files = sorted(root.rglob("*.ply"))

print("Found", len(ply_files), ".ply files")
for p in ply_files[:30]:
    print(p)


CWD: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/notebooks
Found 0 .ply files
